# Week 32, Day 2: Finite cables, branching trees and Rall's equivalent cylinder

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w032/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Input resistance from compartments

Write `R_inf(d_um)`, the input resistance (Ω) of a semi-infinite cable. Then write `cable_rin(d_um, L_cm)`, which builds the conductance matrix of a sealed cable of 200 compartments, injects 1 nA into the first, and returns V/I. Compare it with R_inf coth(L/λ) for a 2 μm cable one length constant long.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rm, ri = 20000.0, 150.0

def lam_cm(d_um):
    return np.sqrt(rm * d_um * 1e-4 / (4 * ri))

def R_inf(d_um):
    return ...

def cable_rin(d_um, L_cm, N=200):
    d, dx = d_um * 1e-4, L_cm / N
    ga, gm = np.pi * d**2 / (4 * ri * dx), np.pi * d * dx / rm
    A = np.diag(np.full(N, gm + 2 * ga)) - ga * np.eye(N, k=1) - ga * np.eye(N, k=-1)
    A[0, 0] -= ga; A[-1, -1] -= ga
    I = np.zeros(N); I[0] = 1e-9
    return ...

for d in (1, 2, 4):
    print(f"d = {d} um: R_inf = {R_inf(d) / 1e6:6.1f} MOhm")
R_sim = cable_rin(2.0, lam_cm(2.0))
print(f"sealed cable of length lambda: compartments {R_sim / 1e6:.1f} MOhm, formula {R_inf(2.0) / np.tanh(1) / 1e6:.1f} MOhm")

In [ ]:
import numpy as np
def test_rinf():
    "R_inf follows the formula, and scales with diameter to the power -3/2"
    assert abs(R_inf(2.0) - 2 / np.pi * np.sqrt(20000 * 150) * (2e-4) ** -1.5) < 1e-3
    assert abs(R_inf(4.0) / R_inf(1.0) - 4 ** -1.5) < 1e-12

def test_sealed():
    "the compartmental input resistance of a sealed cable of length lambda matches R_inf coth(1) within 1%"
    assert abs(R_sim - R_inf(2.0) / np.tanh(1.0)) / R_sim < 0.01

test_rinf()
test_sealed()
print("All checks passed")

### Exercise: Testing the equivalent cylinder

`tree_rin(segments)` builds a compartmental model of any tree given as a list of (diameter in μm, length in cm, parent index) and returns the input resistance at the root. Compute the daughter diameter `d_rall` that satisfies the rule for a 4 μm parent, then compare the input resistance of a tree (parent of length 0.6 λ, two daughters each 0.6 of their own λ) with a cylinder of the parent's diameter and length 1.2 λ, for daughters of `d_rall`, 1.5 and 3.5 μm.

In [ ]:
def tree_rin(segments, n_per=50):
    gm, edges, last = [], [], {}
    for s, (d_um, L, parent) in enumerate(segments):
        d, dx, start = d_um * 1e-4, L / n_per, len(gm)
        for j in range(n_per):
            gm.append(np.pi * d * dx / rm)
            if j > 0:
                edges.append((start + j - 1, start + j, np.pi * d**2 / (4 * ri * dx)))
        if parent is not None:
            pd, pL, _ = segments[parent]
            r = 4 * ri * (pL / n_per / 2) / (np.pi * (pd * 1e-4) ** 2) + 4 * ri * (dx / 2) / (np.pi * d**2)
            edges.append((last[parent], start, 1 / r))
        last[s] = start + n_per - 1
    A = np.diag(gm)
    for i, j, g in edges:
        A[i, i] += g; A[j, j] += g; A[i, j] -= g; A[j, i] -= g
    I = np.zeros(len(gm)); I[0] = 1e-9
    return np.linalg.solve(A, I)[0] / 1e-9

d_rall = ...
rin_cyl = tree_rin([(4.0, 1.2 * lam_cm(4.0), None)])
rin_tree = {}
for dd in (round(d_rall, 3), 1.5, 3.5):
    tree = [(4.0, 0.6 * lam_cm(4.0), None), (dd, 0.6 * lam_cm(dd), 0), (dd, 0.6 * lam_cm(dd), 0)]
    rin_tree[dd] = ...
    print(f"daughters {dd:5.3f} um (sum of d^1.5 / parent's = {2 * dd ** 1.5 / 4 ** 1.5:.2f}): tree {rin_tree[dd] / 1e6:6.1f} MOhm, cylinder {rin_cyl / 1e6:6.1f} MOhm")

In [ ]:
import numpy as np
def test_rall():
    "d_rall satisfies the three-halves rule for two equal daughters of a 4 um parent"
    assert abs(2 * d_rall ** 1.5 - 4.0 ** 1.5) < 1e-9

def test_match():
    "with Rall's diameter the tree matches the cylinder within 1%, with 1.5 um daughters it differs by more than 10%"
    assert abs(rin_tree[round(d_rall, 3)] / rin_cyl - 1) < 0.01 and abs(rin_tree[1.5] / rin_cyl - 1) > 0.1

test_rall()
test_match()
print("All checks passed")